## 1. Configuration

In [1]:
import os, warnings, numpy as np, pandas as pd, re
from sklearn.model_selection import StratifiedKFold
from lifelines import CoxPHFitter
import torch, torch.nn as nn
warnings.filterwarnings("ignore")
SEED=42; np.random.seed(SEED); torch.manual_seed(SEED)

BASE = r"C:\Users\fatma\OneDrive\Desktop\PFE"
CROSSWALK_CSV = os.path.join(BASE, "final_crosswalk", "crosswalk_EXPANDED.csv")
GENOMIC_FEAT  = os.path.join(BASE, "final_crosswalk", "genomic_features_matched_FINAL.csv")  # 122 with full features
IMAGING_CSV   = os.path.join(BASE, "tcga_features_with_clinical.csv")
OUT_DIR       = os.path.join(BASE, "fusion_expanded"); os.makedirs(OUT_DIR, exist_ok=True)

SUBTYPES=["Astrocytoma","Oligodendroglioma","Glioblastoma"]
PUBLISHED={"Astrocytoma":0.655,"Oligodendroglioma":0.688,"Glioblastoma":0.588}
OUTER_FOLDS, N_REPEATS = 5, 10
PENALIZER, L1_RATIO = 0.1, 0.5     # elastic-net (lab framework); ridge fallback if a fold diverges

for p in [CROSSWALK_CSV, IMAGING_CSV]:
    print(f"{'ok ' if os.path.isfile(p) else 'MISSING'}  {p}")

ok   C:\Users\fatma\OneDrive\Desktop\PFE\final_crosswalk\crosswalk_EXPANDED.csv
ok   C:\Users\fatma\OneDrive\Desktop\PFE\tcga_features_with_clinical.csv


## 2. Build the aligned table

In [2]:
def norm(x):
    s=str(x).strip().upper(); m=re.match(r"(TCGA-[A-Z0-9]{2}-[A-Z0-9]{4})",s); return m.group(1) if m else s

cross=pd.read_csv(CROSSWALK_CSV); cross["tcga_id"]=cross.tcga_id.map(norm)
img=pd.read_csv(IMAGING_CSV);     img["tcga_id"]=img.tcga_id.map(norm)

DROP={"tcga_id","subtype","os_days","os_event","mask_source","brats_folder","collection",
      "mgmt_mapping","residual_vol","status","folder","multifocal","n_components","ncr_et_ratio",
      "subtype_idx"}
MRI_FEATS=[c for c in img.columns if c not in DROP and pd.api.types.is_numeric_dtype(img[c])]

df=(cross[["tcga_id","genomic_risk_score","match_method","subtype","os_days","os_event"]]
    .merge(img[["tcga_id"]+MRI_FEATS], on="tcga_id", how="inner").drop_duplicates("tcga_id"))
df=df[df.os_days.notna()&(df.os_days>0)&df.os_event.isin([0,1])&df.subtype.isin(SUBTYPES)].reset_index(drop=True)
df["subtype_idx"]=df.subtype.map({s:i for i,s in enumerate(SUBTYPES)}).astype(int)
df["has_full_genomic"]=df.match_method.ne("computed_score")   # ViT-capable subset

print(f"cohort: {len(df)} patients, {int(df.os_event.sum())} events")
print(df.groupby("subtype").agg(n=("os_event","size"),events=("os_event","sum")).reindex(SUBTYPES).to_string())
print(f"\nfeature-complete (ViT-capable): {int(df.has_full_genomic.sum())}")
print(f"MRI features ({len(MRI_FEATS)}): {MRI_FEATS}")

cohort: 165 patients, 72 events
                    n  events
subtype                      
Astrocytoma        57      10
Oligodendroglioma  27       5
Glioblastoma       81      57

feature-complete (ViT-capable): 122
MRI features (14): ['et_volume', 'ncr_volume', 'ed_volume', 'wt_volume', 'tc_volume', 'ce_rim_width', 'surface_regularity', 'max_diameter', 'solidity', 'sphericity', 'et_wt_ratio', 'ncr_et_ratio_log', 'centroid_dev', 'low_enhancement']


## 3. Metrics + genomic-score check

In [3]:
def c_index(t,e,r):
    t=np.asarray(t,float);e=np.asarray(e,int);r=np.asarray(r,float)
    num=den=0.0
    for i in range(len(t)):
        if e[i]!=1: continue
        L=t>t[i]; den+=L.sum(); num+=(r[i]>r[L]).sum()+0.5*(r[i]==r[L]).sum()
    return num/den if den>0 else np.nan
def _c_pairs(t,e,r):
    """concordant, total pairs (for pooling across strata)"""
    t=np.asarray(t,float);e=np.asarray(e,int);r=np.asarray(r,float)
    num=den=0.0
    for i in range(len(t)):
        if e[i]!=1: continue
        L=t>t[i]; den+=L.sum(); num+=(r[i]>r[L]).sum()+0.5*(r[i]==r[L]).sum()
    return num,den

def c_within(t,e,r,s):
    """Event-weighted stratified C-index: concordance computed WITHIN each subtype,
    then pooled by event weight (not averaged). Far more stable than equal-weight
    averaging when strata differ greatly in event count (GBM 57 vs Oligo 5)."""
    N=D=0.0
    for u in np.unique(s):
        m=s==u
        if m.sum()>4 and e[m].sum()>=2:
            n,d=_c_pairs(t[m],e[m],r[m]); N+=n; D+=d
    return float(N/D) if D>0 else np.nan

print("Genomic-score check (per subtype):")
for st in SUBTYPES:
    s=df[df.subtype==st]
    if s.os_event.sum()>=3:
        c=c_index(s.os_days,s.os_event,s.genomic_risk_score)
        print(f"  {st:20} C={c:.3f}  published {PUBLISHED[st]:.3f}")

Genomic-score check (per subtype):
  Astrocytoma          C=0.609  published 0.655
  Oligodendroglioma    C=0.569  published 0.688
  Glioblastoma         C=0.575  published 0.588


## 4. The four arms (elastic-net + ridge fallback)

In [4]:
def _fit_cox(dframe, tr, cols, strat):
    """Elastic-net Cox; if it diverges, retry ridge. Drops constant strata."""
    d=dframe.copy()
    use_strata = strat if (strat and d.iloc[tr][strat[0]].nunique()>1) else None
    if use_strata is None and strat: d=d.drop(columns=strat)
    for pen,l1 in [(PENALIZER,L1_RATIO),(1.0,0.0)]:     # elastic-net, then ridge fallback
        try:
            return CoxPHFitter(penalizer=pen,l1_ratio=l1).fit(d.iloc[tr],"T","E",strata=use_strata), d
        except Exception:
            continue
    return None, d

def fit_mri(tr, X, t, e, s, cols):
    d=pd.DataFrame(X,columns=cols); d["T"]=t; d["E"]=e; d["S"]=s
    mu,sd=d.iloc[tr,:len(cols)].mean(),d.iloc[tr,:len(cols)].std()+1e-8
    d.iloc[:,:len(cols)]=(d.iloc[:,:len(cols)]-mu)/sd
    f,dd=_fit_cox(d,tr,cols,["S"])
    if f is None: return None
    return np.log(f.predict_partial_hazard(dd).values+1e-12)

class FusionCox(nn.Module):
    """Cross-attention fusion, matched to BothEvals_v2 architecture: d=16, 2 heads,
    dropout 0.2, extra dense layer in head -> ~1697 params."""
    def __init__(self,n_mri,d=16):
        super().__init__()
        self.mri=nn.Sequential(nn.Linear(n_mri,d),nn.ReLU(),nn.LayerNorm(d))
        self.gen=nn.Sequential(nn.Linear(1,d),nn.ReLU(),nn.LayerNorm(d))
        self.attn=nn.MultiheadAttention(d,2,batch_first=True)
        self.head=nn.Sequential(nn.Linear(d,d),nn.ReLU(),nn.Dropout(0.2),nn.Linear(d,1))
    def forward(self,xm,xg):
        tok=torch.cat([self.mri(xm).unsqueeze(1),self.gen(xg).unsqueeze(1)],1)
        a,_=self.attn(tok,tok,tok); return self.head(a.mean(1)).squeeze(-1)

def cox_loss(risk,t,e):
    o=torch.argsort(t,descending=True); r=risk[o]; ev=e[o]
    return -((r-torch.logcumsumexp(r,0))*ev).sum()/(ev.sum()+1e-8)

def fit_vit(tr,Xn,gn,t,e,epochs=150):
    net=FusionCox(Xn.shape[1]); opt=torch.optim.Adam(net.parameters(),lr=1e-2,weight_decay=1e-2)
    xm=torch.tensor(Xn[tr],dtype=torch.float32); xg=torch.tensor(gn[tr],dtype=torch.float32).unsqueeze(1)
    tt=torch.tensor(t[tr],dtype=torch.float32); ee=torch.tensor(e[tr],dtype=torch.float32)
    net.train()
    for _ in range(epochs):
        opt.zero_grad(); loss=cox_loss(net(xm,xg),tt,ee)
        if torch.isfinite(loss): loss.backward(); opt.step()
    net.eval()
    with torch.no_grad():
        return net(torch.tensor(Xn,dtype=torch.float32),
                   torch.tensor(gn,dtype=torch.float32).unsqueeze(1)).numpy()

# ---- Pretrained + ensembled ViT (honest best-effort learned fusion) ----
class GenomicAE(nn.Module):
    """Autoencoder to pre-train a genomic embedding on ALL genomic patients (unsupervised).
    Uses more data than the matched cohort, without leakage (no outcomes involved)."""
    def __init__(self,nin,d=8):
        super().__init__()
        self.enc=nn.Sequential(nn.Linear(nin,32),nn.ReLU(),nn.Linear(32,d))
        self.dec=nn.Sequential(nn.Linear(d,32),nn.ReLU(),nn.Linear(32,nin))
    def forward(self,x): z=self.enc(x); return self.dec(z),z

class PretrainedFusionCox(nn.Module):
    """ViT using a FROZEN pre-trained genomic embedding + MRI, cross-attention."""
    def __init__(self,n_mri,ae,d=8):
        super().__init__()
        self.ae_enc=ae.enc
        for p in self.ae_enc.parameters(): p.requires_grad=False
        self.mri=nn.Sequential(nn.Linear(n_mri,d),nn.ReLU(),nn.LayerNorm(d))
        self.genln=nn.LayerNorm(d)
        self.attn=nn.MultiheadAttention(d,1,batch_first=True)
        self.head=nn.Sequential(nn.Dropout(0.3),nn.Linear(d,1))
    def forward(self,xm,xg):
        tok=torch.cat([self.mri(xm).unsqueeze(1),self.genln(self.ae_enc(xg)).unsqueeze(1)],1)
        a,_=self.attn(tok,tok,tok); return self.head(a.mean(1)).squeeze(-1)

def pretrain_genomic_ae(Gmatrix, d=8, epochs=100):
    """Gmatrix: (n_patients x n_genomic_features) for ALL available genomic patients."""
    ae=GenomicAE(Gmatrix.shape[1],d)
    opt=torch.optim.Adam(ae.parameters(),lr=1e-3)
    mu,sd=Gmatrix.mean(0),Gmatrix.std(0)+1e-8
    Gt=torch.tensor((Gmatrix-mu)/sd,dtype=torch.float32)
    for _ in range(epochs):
        opt.zero_grad(); rec,_=ae(Gt); ((rec-Gt)**2).mean().backward(); opt.step()
    return ae,(mu,sd)

def fit_vit_pretrained(tr, Xn, Gn, t, e, ae, n_seeds=5, epochs=80):
    """Ensemble of pretrained-embedding ViTs; returns averaged risk for all rows."""
    preds=[]
    for seed in range(n_seeds):
        torch.manual_seed(seed)
        net=PretrainedFusionCox(Xn.shape[1],ae)
        opt=torch.optim.Adam([p for p in net.parameters() if p.requires_grad],lr=5e-3,weight_decay=3e-2)
        xm=torch.tensor(Xn[tr],dtype=torch.float32); xg=torch.tensor(Gn[tr],dtype=torch.float32)
        tt=torch.tensor(t[tr],dtype=torch.float32); ee=torch.tensor(e[tr],dtype=torch.float32)
        net.train()
        for _ in range(epochs):
            opt.zero_grad(); loss=cox_loss(net(xm,xg),tt,ee)
            if torch.isfinite(loss): loss.backward(); opt.step()
        net.eval()
        with torch.no_grad():
            preds.append(net(torch.tensor(Xn,dtype=torch.float32),
                             torch.tensor(Gn,dtype=torch.float32)).numpy())
    return np.mean(preds,axis=0)

def run_ladder(sub_df, label, stratified=True, include_vit=True):
    X=sub_df[MRI_FEATS].astype(float).fillna(0).values
    g=sub_df["genomic_risk_score"].astype(float).values
    t=sub_df["os_days"].astype(float).values
    e=sub_df["os_event"].astype(int).values
    s=sub_df["subtype_idx"].astype(int).values
    vit_ok=sub_df["has_full_genomic"].values
    n=len(t)
    arms={k:[] for k in ["genomic_only","mri_only","score_fusion","vit_fusion"]}
    strat=(s*2+e) if stratified else e
    for rep in range(N_REPEATS):
        for tr,te in StratifiedKFold(OUTER_FOLDS,shuffle=True,random_state=SEED+rep).split(np.zeros(n),strat):
            if e[te].sum()<2 or e[tr].sum()<5: continue
            score=lambda r:(c_within(t[te],e[te],r[te],s[te]) if stratified else c_index(t[te],e[te],r[te]))
            arms["genomic_only"].append(score(g))
            mri_r=fit_mri(tr,X,t,e,s,MRI_FEATS)
            if mri_r is not None: arms["mri_only"].append(score(mri_r))
            if mri_r is not None:
                dd=pd.DataFrame({"g":g,"m":mri_r,"T":t,"E":e,"S":s})
                f,dd2=_fit_cox(dd,tr,["g","m"],["S"] if stratified else None)
                if f is not None:
                    sf=np.log(f.predict_partial_hazard(dd2).values+1e-12); arms["score_fusion"].append(score(sf))
                else:
                    gz=(g-g[tr].mean())/(g[tr].std()+1e-8); mz=(mri_r-mri_r[tr].mean())/(mri_r[tr].std()+1e-8)
                    arms["score_fusion"].append(score(gz+mz))
            # ViT: only train/test on feature-complete patients
            if include_vit:
                trv=tr[vit_ok[tr]]; tev=te[vit_ok[te]]
                if e[trv].sum()>=5 and e[tev].sum()>=2:
                    mu,sd=X[trv].mean(0),X[trv].std(0)+1e-8; Xn=((X-mu)/sd).astype(np.float32)
                    gm,gs=g[trv].mean(),g[trv].std()+1e-8; gn=((g-gm)/gs).astype(np.float32)
                    vit=fit_vit(trv,Xn,gn,t,e)
                    v=(c_within(t[tev],e[tev],vit[tev],s[tev]) if stratified else c_index(t[tev],e[tev],vit[tev]))
                    if np.isfinite(v): arms["vit_fusion"].append(v)
    print(f"\n{'='*58}\n{label}  (n={n}, events={int(e.sum())})\n{'='*58}")
    res={}
    for k in ["genomic_only","mri_only","score_fusion","vit_fusion"]:
        v=np.array(arms[k],float); res[k]=(np.nanmean(v),np.nanstd(v))
        tag=" [122-subset]" if k=="vit_fusion" else ""
        print(f"  {k:16}: C = {np.nanmean(v):.4f} +/- {np.nanstd(v):.4f}  (folds={int(np.isfinite(v).sum())}){tag}")
    print(f"\n  MRI adds to genomics : {res['score_fusion'][0]-res['genomic_only'][0]:+.4f}")
    print(f"  ViT beats score-fus. : {res['vit_fusion'][0]-res['score_fusion'][0]:+.4f}  (on 122-subset)")
    return res

## 5. Primary — Glioblastoma (57 events)

In [5]:
gbm=df[df.subtype=="Glioblastoma"].reset_index(drop=True)
res_gbm=run_ladder(gbm,"GBM (primary)",stratified=False)
pd.DataFrame(res_gbm,index=["mean","std"]).T.to_csv(os.path.join(OUT_DIR,"fusion_gbm.csv"))


GBM (primary)  (n=81, events=57)
  genomic_only    : C = 0.5673 +/- 0.1161  (folds=50)
  mri_only        : C = 0.6006 +/- 0.1039  (folds=50)
  score_fusion    : C = 0.6065 +/- 0.0928  (folds=50)
  vit_fusion      : C = 0.5097 +/- 0.2153  (folds=49) [122-subset]

  MRI adds to genomics : +0.0392
  ViT beats score-fus. : -0.0968  (on 122-subset)


## 6. Pooled — subtype-stratified (72 events)

In [6]:
res_pool=run_ladder(df,"POOLED subtype-stratified",stratified=True)
pd.DataFrame(res_pool,index=["mean","std"]).T.to_csv(os.path.join(OUT_DIR,"fusion_pooled.csv"))


POOLED subtype-stratified  (n=165, events=72)
  genomic_only    : C = 0.5692 +/- 0.1153  (folds=50)
  mri_only        : C = 0.6180 +/- 0.0952  (folds=50)
  score_fusion    : C = 0.6166 +/- 0.0956  (folds=50)
  vit_fusion      : C = 0.6349 +/- 0.1550  (folds=50) [122-subset]

  MRI adds to genomics : +0.0474
  ViT beats score-fus. : +0.0183  (on 122-subset)


## 6c. Score-based ViT — pooled-train / GBM & all-glioma eval

Run immediately after the pooled ladder (6) so this reproduces the same RNG
trajectory as `BothEvals_v2`'s equivalent step, before any later cells (sensitivity
re-run, genomic-AE pretrain) can shift it.

In [7]:
def _full_Gn(d):
    n=len(d); Gn=np.zeros((n,Gall.shape[1]),dtype=np.float32); fm=np.zeros(n,bool)
    for i in range(n):
        si=id2si.get(d.iloc[i].tcga_id,-1)
        if si in sidx and d.iloc[i].has_full_genomic:
            Gn[i]=(Gall[sidx[si]]-g_mu)/g_sd; fm[i]=True
    return Gn,fm


def run_both(df_all, vit_mode, eval_mode, label):
    """vit_mode: 'score' | 'full'.  eval_mode: 'gbm' | 'all'. Pooled training throughout."""
    d=df_all.reset_index(drop=True)
    X=d[MRI_FEATS].astype(float).fillna(0).values
    g=d["genomic_risk_score"].astype(float).values
    t=d["os_days"].astype(float).values; e=d["os_event"].astype(int).values
    s=d["subtype_idx"].astype(int).values
    vit_ok=d["has_full_genomic"].values
    is_gbm=(d["subtype"]=="Glioblastoma").values
    n=len(t)
    Gn_full,full_mask=_full_Gn(d) if vit_mode=="full" else (None,vit_ok)
    arms={k:[] for k in ["genomic_only","mri_only","score_fusion","vit_fusion"]}
    strat=s*2+e
    for rep in range(N_REPEATS):
        for tr,te in StratifiedKFold(OUTER_FOLDS,shuffle=True,random_state=SEED+rep).split(np.zeros(n),strat):
            te_ev = te[is_gbm[te]] if eval_mode=="gbm" else te
            if len(te_ev)==0 or e[te_ev].sum()<2 or e[tr].sum()<5: continue
            sc=(lambda r: c_index(t[te_ev],e[te_ev],r[te_ev])) if eval_mode=="gbm" \
               else (lambda r: c_within(t[te_ev],e[te_ev],r[te_ev],s[te_ev]))
            arms["genomic_only"].append(sc(g))
            mri_r=fit_mri(tr,X,t,e,s,MRI_FEATS)
            if mri_r is not None:
                arms["mri_only"].append(sc(mri_r))
                dd=pd.DataFrame({"g":g,"m":mri_r,"T":t,"E":e,"S":s})
                f,dd2=_fit_cox(dd,tr,["g","m"],["S"])
                if f is not None:
                    sf=np.log(f.predict_partial_hazard(dd2).values+1e-12); arms["score_fusion"].append(sc(sf))
                else:
                    gz=(g-g[tr].mean())/(g[tr].std()+1e-8); mz=(mri_r-mri_r[tr].mean())/(mri_r[tr].std()+1e-8)
                    arms["score_fusion"].append(sc(gz+mz))
            trv=tr[full_mask[tr]]; tev=te_ev[full_mask[te_ev]]
            if e[trv].sum()>=5 and len(tev)>0 and e[tev].sum()>=2:
                mu,sd=X[trv].mean(0),X[trv].std(0)+1e-8; Xn=((X-mu)/sd).astype(np.float32)
                if vit_mode=="score":
                    gm,gs=g[trv].mean(),g[trv].std()+1e-8; gn=((g-gm)/gs).astype(np.float32)
                    risk=fit_vit(trv,Xn,gn,t,e)
                else:
                    risk=fit_vit_pretrained(trv,Xn,Gn_full,t,e,ae,n_seeds=5)
                v=(c_index(t[tev],e[tev],risk[tev]) if eval_mode=="gbm"
                   else c_within(t[tev],e[tev],risk[tev],s[tev]))
                if np.isfinite(v): arms["vit_fusion"].append(v)
    res={k:np.nanmean(np.array(arms[k],float)) for k in arms}
    sd_={k:np.nanstd(np.array(arms[k],float)) for k in arms}
    bs=max(res["genomic_only"],res["mri_only"])
    print(f"\n{'='*60}\n{label}\n{'='*60}")
    for k in ["genomic_only","mri_only","score_fusion","vit_fusion"]:
        v=np.array(arms[k],float)
        print(f"  {k:14}: C = {np.nanmean(v):.4f} +/- {np.nanstd(v):.4f}  (folds={int(np.isfinite(v).sum())})")
    print(f"  ViT vs best single: {res['vit_fusion']-bs:+.4f}  | ViT vs score_fusion: {res['vit_fusion']-res['score_fusion']:+.4f}")
    return res

print("\n########## SCORE-BASED ViT (1 genomic score) ##########")
r_s_gbm = run_both(df,"score","gbm", "SCORE ViT  /  pooled-train  /  GBM eval")
r_s_all = run_both(df,"score","all", "SCORE ViT  /  pooled-train  /  ALL-GLIOMA eval")



########## SCORE-BASED ViT (1 genomic score) ##########

SCORE ViT  /  pooled-train  /  GBM eval
  genomic_only  : C = 0.5673 +/- 0.1161  (folds=50)
  mri_only      : C = 0.6169 +/- 0.0988  (folds=50)
  score_fusion  : C = 0.6154 +/- 0.0989  (folds=50)
  vit_fusion    : C = 0.6332 +/- 0.1708  (folds=49)
  ViT vs best single: +0.0163  | ViT vs score_fusion: +0.0177

SCORE ViT  /  pooled-train  /  ALL-GLIOMA eval
  genomic_only  : C = 0.5692 +/- 0.1153  (folds=50)
  mri_only      : C = 0.6180 +/- 0.0952  (folds=50)
  score_fusion  : C = 0.6166 +/- 0.0956  (folds=50)
  vit_fusion    : C = 0.6357 +/- 0.1484  (folds=50)
  ViT vs best single: +0.0177  | ViT vs score_fusion: +0.0191


## 8b. Feature-selection fix — remove the subtype/survival confound

The original screen ranked genes by raw `|corr|` with pooled `OS.time`. Cancer
subtype is a major confound here: Glioblastoma has a ~67% event rate and ~357-day
median OS, vs. ~21–26% and ~940–965 days for Astrocytoma/Oligodendroglioma, and
genome-wide methylation differs enormously by subtype (it's the basis of the
companion classifier). So a raw pooled correlation screen mostly picks up genes that
track subtype, not genes that predict survival *within* a subtype — confirmed
empirically: stratified held-out testing on the old top-50 showed ~49/50 of them
collapse to non-significance once subtype is controlled for.

**Fix:** rank genes by `|corr|` with the martingale residuals of a
`cancer_type`-stratified null Cox model, instead of raw `OS.time`. This is the same
correction the reference paper's own pooled model uses (`stratifySurv` by cancer
type) and mirrors its SIS screening step (martingale-residual correlation), rather
than inventing a new criterion. Everything downstream (AE pretraining, ViT fusion,
CV, the K sweep) is unchanged — only the feature-selection criterion itself changes,
isolating its effect on the results.


In [8]:
def stratified_martingale_residuals(frame, time_col="OS.time", event_col="OS", strata_col="cancer_type"):
    """Null Cox model stratified by cancer_type -> martingale residuals with the subtype
    baseline-hazard effect removed. Correlating features against these instead of raw
    OS.time removes the subtype/survival confound from the selection step."""
    null_df = frame[[time_col, event_col, strata_col]].dropna().copy()
    cph_null = CoxPHFitter()
    cph_null.fit(null_df, duration_col=time_col, event_col=event_col, strata=[strata_col])
    resid = cph_null.compute_residuals(null_df, kind="martingale")
    return resid["martingale"].reindex(frame.index)

def select_features_stratified(frame, gen_feats, k, time_col="OS.time", event_col="OS", strata_col="cancer_type"):
    """Drop-in replacement for the old raw-|corr|-with-OS.time screen. Same top-k-by-
    |correlation| logic and same low-event fallback to top-variance, just correlated
    against subtype-adjusted martingale residuals instead of raw survival time."""
    ev = frame[event_col].astype(int).values
    mart = stratified_martingale_residuals(frame, time_col, event_col, strata_col)
    valid = mart.notna().values
    if ev[valid].sum() >= 20:
        G = frame.loc[valid, gen_feats].fillna(frame.loc[valid, gen_feats].mean()).values.astype(np.float32)
        m = mart.values[valid].astype(np.float32)
        Gc, mc = G - G.mean(0), m - m.mean()
        num = (Gc * mc[:, None]).sum(0)
        den = np.sqrt((Gc**2).sum(0) * (mc**2).sum())
        corr = np.divide(num, den, out=np.zeros_like(num), where=den > 0)
        sel_idx = np.argsort(-np.abs(corr))[:k]
        print(f"selected {len(sel_idx)} / {len(gen_feats)} genomic features "
              f"(|corr| with subtype-stratified martingale residuals, {int(ev[valid].sum())} observed deaths)")
    else:
        Graw = frame[gen_feats].fillna(frame[gen_feats].mean()).values.astype(np.float32)
        sel_idx = np.argsort(-Graw.var(0))[:k]
        print(f"too few observed deaths for correlation screen -- fell back to top-variance {k} features")
    return [gen_feats[i] for i in sel_idx]


## 9. Pretrained + ensembled ViT (best-effort learned fusion)

This is the strongest honest learned-fusion attempt: the genomic embedding is **pre-trained on all
available genomic patients** (unsupervised autoencoder — uses more data without leakage), then a
cross-attention ViT is trained on the matched cohort and **ensembled over 5 seeds** to reduce
variance. Runs only on the feature-complete subset (patients with the full 34,099-feature matrix),
compared against score_fusion on the *same* subset.

Report its cross-validated mean and spread honestly — do not cherry-pick a seed.

In [9]:
# load the full genomic feature matrix (for pre-training + feature-complete patients)
GEN_MATRIX = os.path.join(BASE, "gene_region_matrix_with_ids_and_risk.csv")
gm_full = pd.read_csv(GEN_MATRIX)
meta=["sample_index","OS","OS.time","cancer_type","genomic_risk_score",
      "survival_365d","survival_730d","survival_1095d"]
GEN_FEATS=[c for c in gm_full.columns if c not in meta]
print(f"genomic feature matrix: {gm_full.shape[0]} patients x {len(GEN_FEATS)} features")
N_SEL_FEATS = 300
SEL_FEATS = select_features_stratified(gm_full, GEN_FEATS, N_SEL_FEATS)   # <-- subtype-stratified selection (was: raw |corr| with pooled OS.time)

# pre-train the genomic autoencoder on ALL 711 genomic patients (unsupervised, no outcomes)
Gall = gm_full[SEL_FEATS].fillna(gm_full[SEL_FEATS].mean()).values.astype(np.float32)
ae, (g_mu,g_sd) = pretrain_genomic_ae(Gall, d=8, epochs=100)
print("genomic autoencoder pre-trained on all patients")

# attach each feature-complete patient's genomic feature vector via sample_index
fc = df[df.has_full_genomic].copy()
sidx = dict(zip(gm_full.sample_index, range(len(gm_full))))
# map crosswalk sample_index -> row in gm_full
cw = pd.read_csv(CROSSWALK_CSV); cw["tcga_id"]=cw.tcga_id.map(norm)
id2si = dict(zip(cw.tcga_id, cw.sample_index))

def run_pretrained_vit(sub_df, label, stratified):
    d = sub_df[sub_df.has_full_genomic].reset_index(drop=True)
    # build genomic feature matrix for these patients
    rows=[]
    keep=[]
    for i,r in d.iterrows():
        si=id2si.get(r.tcga_id,-1)
        if si in sidx:
            rows.append(Gall[sidx[si]]); keep.append(i)
    if len(keep)<20:
        print(f"{label}: too few feature-complete patients ({len(keep)}) for ViT"); return
    d=d.loc[keep].reset_index(drop=True)
    G=np.array(rows,dtype=np.float32); Gn=((G-g_mu)/g_sd).astype(np.float32)
    X=d[MRI_FEATS].astype(float).fillna(0).values
    t=d.os_days.values.astype(float); e=d.os_event.values.astype(int); s=d.subtype_idx.values.astype(int)
    n=len(t); strat=(s*2+e) if stratified else e
    vit=[]; sf=[]
    for rep in range(N_REPEATS):
        for tr,te in StratifiedKFold(OUTER_FOLDS,shuffle=True,random_state=SEED+rep).split(np.zeros(n),strat):
            if e[te].sum()<2 or e[tr].sum()<5: continue
            mu,sd=X[tr].mean(0),X[tr].std(0)+1e-8; Xn=((X-mu)/sd).astype(np.float32)
            r=fit_vit_pretrained(tr,Xn,Gn,t,e,ae,n_seeds=5)
            sc=(c_within(t[te],e[te],r[te],s[te]) if stratified else c_index(t[te],e[te],r[te]))
            if np.isfinite(sc): vit.append(sc)
    v=np.array(vit,float)
    print(f"\n{label} pretrained+ensemble ViT (n={n} feature-complete):")
    print(f"  C = {np.nanmean(v):.4f} +/- {np.nanstd(v):.4f}  (folds={int(np.isfinite(v).sum())})")
    return np.nanmean(v),np.nanstd(v)

print("\n=== GBM ===");    run_pretrained_vit(df[df.subtype=="Glioblastoma"],"GBM",False)
print("\n=== POOLED ==="); run_pretrained_vit(df,"POOLED",True)

genomic feature matrix: 711 patients x 34099 features
selected 300 / 34099 genomic features (|corr| with subtype-stratified martingale residuals, 256 observed deaths)
genomic autoencoder pre-trained on all patients

=== GBM ===

GBM pretrained+ensemble ViT (n=38 feature-complete):
  C = 0.5784 +/- 0.1357  (folds=50)

=== POOLED ===

POOLED pretrained+ensemble ViT (n=122 feature-complete):
  C = 0.6348 +/- 0.1346  (folds=50)


(np.float64(0.6348383395436767), np.float64(0.13460453528567776))

## 10. Full-feature ViT — feature-count sweep

Repeats cell 15's exact feature-selection + genomic-AE pretrain logic and cell 17's
exact full-feature ViT calls (`run_both(..., "full", ...)`), looping `N_SEL_FEATS`
over 50, 72, 100, 300, 1000, 5000. Nothing upstream is changed.

**Reproducibility fix:** `pretrain_genomic_ae` initializes `GenomicAE`'s weights from
whatever the *global* PyTorch RNG state happens to be at call time. Looping over
`N_SEL_FEATS` means each iteration's AE init silently inherits RNG state left over
from every previous iteration's training — so results for a given `N_SEL_FEATS`
depend on where it sits in the list, not just on the feature count. We now reseed to
`SEED` immediately before each `pretrain_genomic_ae` call so every feature-count arm
gets the same, independent AE initialization, decoupled from loop position and from
whatever ran earlier in the notebook.

**Feature-selection fix (section 8b):** this sweep now selects features via `select_features_stratified` (subtype-adjusted martingale-residual correlation) instead of raw pooled `|corr|` with `OS.time`. Results are not comparable to the earlier unstratified sweep — re-run this cell fresh rather than diffing against old numbers.


In [10]:
N_SEL_FEATS_LIST = [50, 72, 100, 300, 1000, 5000]
full_feature_results = {}

for N_SEL_FEATS in N_SEL_FEATS_LIST:
    # --- subtype-stratified selection (was: raw |corr| with pooled OS.time) ---
    SEL_FEATS = select_features_stratified(gm_full, GEN_FEATS, N_SEL_FEATS)

    # --- exact cell-15 AE pretrain logic ---
    Gall = gm_full[SEL_FEATS].fillna(gm_full[SEL_FEATS].mean()).values.astype(np.float32)
    torch.manual_seed(SEED)          # <-- reproducibility fix: pin AE init, decoupled from loop position
    ae, (g_mu,g_sd) = pretrain_genomic_ae(Gall, d=8, epochs=100)
    print(f"genomic autoencoder pre-trained on all patients (N_SEL_FEATS={N_SEL_FEATS})")

    # --- exact cell-17 full-feature ViT calls ---
    print(f"\n########## FULL-FEATURE ViT ({N_SEL_FEATS} features) ##########")
    r_f_gbm = run_both(df, "full", "gbm", f"FULL-FEATURE ViT ({N_SEL_FEATS} feats)  /  pooled-train  /  GBM eval")
    r_f_all = run_both(df, "full", "all", f"FULL-FEATURE ViT ({N_SEL_FEATS} feats)  /  pooled-train  /  ALL-GLIOMA eval")

    full_feature_results[N_SEL_FEATS] = {"gbm": r_f_gbm, "all": r_f_all}

print("\n"+"="*70)
print("SUMMARY: Full-feature ViT C-index across feature-count sweep")
print("="*70)
print(f"{'N_SEL_FEATS':<12}{'GBM eval':>14}{'all-glioma':>14}")
for n in N_SEL_FEATS_LIST:
    print(f"{n:<12}{full_feature_results[n]['gbm']['vit_fusion']:>14.4f}{full_feature_results[n]['all']['vit_fusion']:>14.4f}")

pd.DataFrame({n: {
    "gbm_vit": full_feature_results[n]["gbm"]["vit_fusion"],
    "all_vit": full_feature_results[n]["all"]["vit_fusion"],
} for n in N_SEL_FEATS_LIST}).T.to_csv(os.path.join(OUT_DIR, "fullfeature_vit_sweep_stratified_selection.csv"))

selected 50 / 34099 genomic features (|corr| with subtype-stratified martingale residuals, 256 observed deaths)
genomic autoencoder pre-trained on all patients (N_SEL_FEATS=50)

########## FULL-FEATURE ViT (50 features) ##########

FULL-FEATURE ViT (50 feats)  /  pooled-train  /  GBM eval
  genomic_only  : C = 0.5673 +/- 0.1161  (folds=50)
  mri_only      : C = 0.6169 +/- 0.0988  (folds=50)
  score_fusion  : C = 0.6154 +/- 0.0989  (folds=50)
  vit_fusion    : C = 0.6350 +/- 0.1843  (folds=49)
  ViT vs best single: +0.0181  | ViT vs score_fusion: +0.0196

FULL-FEATURE ViT (50 feats)  /  pooled-train  /  ALL-GLIOMA eval
  genomic_only  : C = 0.5692 +/- 0.1153  (folds=50)
  mri_only      : C = 0.6180 +/- 0.0952  (folds=50)
  score_fusion  : C = 0.6166 +/- 0.0956  (folds=50)
  vit_fusion    : C = 0.6190 +/- 0.1789  (folds=50)
  ViT vs best single: +0.0010  | ViT vs score_fusion: +0.0024
selected 72 / 34099 genomic features (|corr| with subtype-stratified martingale residuals, 256 observed 

In [12]:
WINNER_KS = [50, 1000]
sel_feats_by_k = {k: select_features_stratified(gm_full, GEN_FEATS, k) for k in WINNER_KS}
for k in WINNER_KS:
    print(f"K={k}: {len(sel_feats_by_k[k])} features captured")

selected 50 / 34099 genomic features (|corr| with subtype-stratified martingale residuals, 256 observed deaths)
selected 1000 / 34099 genomic features (|corr| with subtype-stratified martingale residuals, 256 observed deaths)
K=50: 50 features captured
K=1000: 1000 features captured


In [13]:
import re

REFERENCE_GENES = {
    "DIAPH1", "LOC100133893", "ATXN7L1", "LRRC14B",
    "POMT1", "HSPG2", "C8ORF4", "PANK3", "GGT1", "FLJ35776", "KIAA0408",
    "EDNRA", "PCBD1", "LRIT2", "WDR70", "HCN3", "ZNF160", "PSMC5", "TRIB1",
    "LASP1", "TTC32", "ZNF287", "IPMK", "MC2R", "ZNF266", "PRMT3", "EDN2",
    "TRMT61B", "PRKAR2A", "LHFPL5", "RDM1", "C20ORF95", "PHOSPHO1", "FAM92B",
    "MIR1260", "KIF17", "DDTL", "SFRP5", "IKZF1", "APOLD1", "PDGFRL", "ACTB",
    "HOXC9", "FAM43B", "NPW", "COL12A1", "GSC2", "HOXA7", "ISG15", "RANGAP1",
    "HOXD11", "UMODL1",
    "HMGA2", "RHOBTB3", "TMEM90B", "CNOT8", "RNF115", "POFUT1", "SPP1", "ARIH1",
    "TRIM4", "ZNF19", "MIR1259", "MIR1292", "C1ORF107", "PF4", "LOC285074",
    "LOC440461", "TPD52L2", "ATHL1", "ANKRD22", "SPAG4",
}

_REGION_SUFFIX = re.compile(r"[_\-\.]+(body|promoter|tss200|tss1500|5utr|1stexon)$", re.IGNORECASE)

def base_gene_symbol(feat_name: str) -> str:
    return _REGION_SUFFIX.sub("", feat_name).upper()

provenance_by_k = {}
for k in WINNER_KS:
    feats = sel_feats_by_k[k]
    matched = [f for f in feats if base_gene_symbol(f) in REFERENCE_GENES]
    print(f"K={k}: {len(matched)}/{len(feats)} already reported in Abulizi & Mamatjan")
    if matched:
        print("   matched:", matched)
    provenance_by_k[k] = matched

K=50: 4/50 already reported in Abulizi & Mamatjan
   matched: ['PDGFRL__body', 'IPMK__body', 'LOC285074__body', 'HOXA7__promoter']
K=1000: 17/1000 already reported in Abulizi & Mamatjan
   matched: ['PDGFRL__body', 'IPMK__body', 'LOC285074__body', 'HOXA7__promoter', 'PANK3__promoter', 'PSMC5__promoter', 'POMT1__promoter', 'RHOBTB3__promoter', 'TRIM4__promoter', 'ZNF287__promoter', 'EDNRA__promoter', 'DIAPH1__body', 'TPD52L2__body', 'POFUT1__promoter', 'EDN2__body', 'MIR1292__promoter', 'ZNF160__promoter']


In [14]:
np.random.seed(SEED)
N_BOOT = 100   # reduced from 200 since each iteration now fits a Cox model, not just correlation
n_patients = len(gm_full)

bootstrap_stability_by_k = {}
for k in WINNER_KS:
    target_set = set(sel_feats_by_k[k])
    counts = {f: 0 for f in target_set}
    for b in range(N_BOOT):
        boot_idx = np.random.choice(n_patients, size=n_patients, replace=True)
        boot_df = gm_full.iloc[boot_idx].reset_index(drop=True)
        if boot_df["OS"].sum() < 20 or boot_df["cancer_type"].nunique() < 2:
            continue
        try:
            boot_sel = set(select_features_stratified(boot_df, GEN_FEATS, k))
        except Exception:
            continue
        for f in target_set:
            if f in boot_sel:
                counts[f] += 1
    df_k = pd.DataFrame({
        "feature": list(target_set),
        "base_gene_symbol": [base_gene_symbol(f) for f in target_set],
        "in_reference_paper": [base_gene_symbol(f) in REFERENCE_GENES for f in target_set],
        "bootstrap_selection_freq": [counts[f] / N_BOOT for f in target_set],
    }).sort_values("bootstrap_selection_freq", ascending=False)
    bootstrap_stability_by_k[k] = df_k
    print(f"\n=== K={k} bootstrap stability (top 15) ===")
    print(df_k.head(15).to_string(index=False))
    df_k.to_csv(os.path.join(OUT_DIR, f"sel_feats_k{k}_stratified_bootstrap.csv"), index=False)

selected 50 / 34099 genomic features (|corr| with subtype-stratified martingale residuals, 250 observed deaths)
selected 50 / 34099 genomic features (|corr| with subtype-stratified martingale residuals, 249 observed deaths)
selected 50 / 34099 genomic features (|corr| with subtype-stratified martingale residuals, 268 observed deaths)
selected 50 / 34099 genomic features (|corr| with subtype-stratified martingale residuals, 264 observed deaths)
selected 50 / 34099 genomic features (|corr| with subtype-stratified martingale residuals, 274 observed deaths)
selected 50 / 34099 genomic features (|corr| with subtype-stratified martingale residuals, 256 observed deaths)
selected 50 / 34099 genomic features (|corr| with subtype-stratified martingale residuals, 278 observed deaths)
selected 50 / 34099 genomic features (|corr| with subtype-stratified martingale residuals, 237 observed deaths)
selected 50 / 34099 genomic features (|corr| with subtype-stratified martingale residuals, 268 observed 

In [15]:
from sklearn.model_selection import train_test_split

np.random.seed(SEED)
N_SPLITS = 20
TEST_FRAC = 0.3
_ev_full = gm_full["OS"].astype(int).values

heldout_by_k = {}
for k in WINNER_KS:
    feats = sel_feats_by_k[k]
    sig_counts = {f: 0 for f in feats}
    valid_splits = 0
    for split_i in range(N_SPLITS):
        idx_all = np.arange(len(gm_full))
        tr_idx, te_idx = train_test_split(
            idx_all, test_size=TEST_FRAC, random_state=SEED + split_i, stratify=_ev_full
        )
        if _ev_full[te_idx].sum() < 10:
            continue
        valid_splits += 1
        for f in feats:
            mu, sd = gm_full.iloc[tr_idx][f].mean(), gm_full.iloc[tr_idx][f].std() + 1e-8
            sub = gm_full.iloc[te_idx][[f, "OS.time", "OS", "cancer_type"]].dropna().rename(columns={f: "x"})
            sub["x"] = (sub["x"] - mu) / sd
            if sub["OS"].sum() < 5 or sub["cancer_type"].nunique() < 2:
                continue
            try:
                cph = CoxPHFitter()
                cph.fit(sub[["x", "OS.time", "OS", "cancer_type"]],
                        duration_col="OS.time", event_col="OS", strata=["cancer_type"])
                if cph.summary.loc["x", "p"] < 0.05:
                    sig_counts[f] += 1
            except Exception:
                pass

    df_k = pd.DataFrame({
        "feature": feats,
        "base_gene_symbol": [base_gene_symbol(f) for f in feats],
        "in_reference_paper": [base_gene_symbol(f) in REFERENCE_GENES for f in feats],
        "bootstrap_selection_freq": [bootstrap_stability_by_k[k].set_index("feature")["bootstrap_selection_freq"].get(f, np.nan) for f in feats],
        "heldout_sig_rate_stratified": [sig_counts[f] / valid_splits for f in feats],
    }).sort_values("heldout_sig_rate_stratified", ascending=False)
    heldout_by_k[k] = df_k
    print(f"\n=== K={k}: valid splits {valid_splits}/{N_SPLITS} ===")
    print(df_k.to_string(index=False))
    df_k.to_csv(os.path.join(OUT_DIR, f"sel_feats_k{k}_stratified_heldout.csv"), index=False)


=== K=50: valid splits 20/20 ===
            feature base_gene_symbol  in_reference_paper  bootstrap_selection_freq  heldout_sig_rate_stratified
         IPMK__body             IPMK                True                      0.76                         1.00
HNRNPA1L2__promoter        HNRNPA1L2               False                      0.17                         1.00
  ANKRD40__promoter          ANKRD40               False                      0.16                         0.90
     PPIB__promoter             PPIB               False                      0.24                         0.90
    CKAP4__promoter            CKAP4               False                      0.31                         0.90
       PDGFRL__body           PDGFRL                True                      0.72                         0.90
        DCTN2__body            DCTN2               False                      0.54                         0.85
  MTERFD1__promoter          MTERFD1               False              

KeyboardInterrupt: 

In [16]:
print(gm_full["cancer_type"].value_counts())

cancer_type
ASTRO    308
OLIGO    210
GBM      193
Name: count, dtype: int64


In [18]:
GBM_LABEL = "GBM"  

gbm_df = gm_full[gm_full["cancer_type"] == GBM_LABEL].reset_index(drop=True)
print(f"GBM-only cohort: {len(gbm_df)} patients, {gbm_df['OS'].sum()} events")

np.random.seed(SEED)
N_SPLITS = 20
TEST_FRAC = 0.3
_ev_gbm = gbm_df["OS"].astype(int).values

feats_k50 = sel_feats_by_k[50]
sig_counts_gbm = {f: 0 for f in feats_k50}
valid_splits_gbm = 0

for split_i in range(N_SPLITS):
    idx_all = np.arange(len(gbm_df))
    tr_idx, te_idx = train_test_split(
        idx_all, test_size=TEST_FRAC, random_state=SEED + split_i, stratify=_ev_gbm
    )
    if _ev_gbm[te_idx].sum() < 8:
        continue
    valid_splits_gbm += 1
    for f in feats_k50:
        mu, sd = gbm_df.iloc[tr_idx][f].mean(), gbm_df.iloc[tr_idx][f].std() + 1e-8
        sub = gbm_df.iloc[te_idx][[f, "OS.time", "OS"]].dropna().rename(columns={f: "x"})
        sub["x"] = (sub["x"] - mu) / sd
        if sub["OS"].sum() < 5:
            continue
        try:
            cph = CoxPHFitter()
            cph.fit(sub[["x", "OS.time", "OS"]], duration_col="OS.time", event_col="OS")
            if cph.summary.loc["x", "p"] < 0.05:
                sig_counts_gbm[f] += 1
        except Exception:
            pass

gbm_only_df = pd.DataFrame({
    "feature": feats_k50,
    "base_gene_symbol": [base_gene_symbol(f) for f in feats_k50],
    "in_reference_paper": [base_gene_symbol(f) in REFERENCE_GENES for f in feats_k50],
    "gbm_only_heldout_sig_rate": [sig_counts_gbm[f] / valid_splits_gbm for f in feats_k50],
}).sort_values("gbm_only_heldout_sig_rate", ascending=False)

print(f"\nvalid splits: {valid_splits_gbm} / {N_SPLITS}\n")
print(gbm_only_df.to_string(index=False))
gbm_only_df.to_csv(os.path.join(OUT_DIR, "sel_feats_k50_gbm_only_heldout.csv"), index=False)

GBM-only cohort: 193 patients, 130 events

valid splits: 20 / 20

            feature base_gene_symbol  in_reference_paper  gbm_only_heldout_sig_rate
        TTRAP__body            TTRAP               False                       0.70
    LOC285074__body        LOC285074                True                       0.60
      PLEKHO1__body          PLEKHO1               False                       0.60
      RSPH10B__body          RSPH10B               False                       0.60
  ANKRD40__promoter          ANKRD40               False                       0.55
HNRNPA1L2__promoter        HNRNPA1L2               False                       0.55
   PRDM15__promoter           PRDM15               False                       0.55
        SEP15__body            SEP15               False                       0.50
    OR1S2__promoter            OR1S2               False                       0.50
  ST3GAL6__promoter          ST3GAL6               False                       0.45
    LOC441

In [20]:
np.random.seed(SEED)
N_PERM = 20          # was 100
N_SPLITS_PERM = 10   # was 20 (only for the permutation loop, not your real result)
perm_max_rates = []

for perm_i in range(N_PERM):
    perm_df = gbm_df.copy()
    shuffle_idx = np.random.permutation(len(perm_df))
    perm_df["OS"] = perm_df["OS"].values[shuffle_idx]
    perm_df["OS.time"] = perm_df["OS.time"].values[shuffle_idx]
    _ev_perm = perm_df["OS"].astype(int).values

    perm_sig_counts = {f: 0 for f in feats_k50}
    valid_perm_splits = 0
    for split_i in range(N_SPLITS_PERM):
        idx_all = np.arange(len(perm_df))
        tr_idx, te_idx = train_test_split(
            idx_all, test_size=TEST_FRAC, random_state=SEED + split_i, stratify=_ev_perm
        )
        if _ev_perm[te_idx].sum() < 8:
            continue
        valid_perm_splits += 1
        for f in feats_k50:
            mu, sd = perm_df.iloc[tr_idx][f].mean(), perm_df.iloc[tr_idx][f].std() + 1e-8
            sub = perm_df.iloc[te_idx][[f, "OS.time", "OS"]].dropna().rename(columns={f: "x"})
            sub["x"] = (sub["x"] - mu) / sd
            if sub["x"].std() < 1e-3 or sub["OS"].sum() < 5:   # <-- near-constant guard, fixes the slowdown
                continue
            try:
                cph = CoxPHFitter()
                cph.fit(sub[["x", "OS.time", "OS"]], duration_col="OS.time", event_col="OS")
                if cph.summary.loc["x", "p"] < 0.05:
                    perm_sig_counts[f] += 1
            except Exception:
                pass
    rates = [perm_sig_counts[f] / valid_perm_splits for f in feats_k50]
    perm_max_rates.append(max(rates))
    print(f"perm {perm_i+1}/{N_PERM} done, best rate this perm: {max(rates):.2f}")

perm_max_rates = np.array(perm_max_rates)
observed_best = 0.70  # TTRAP
p_value = (perm_max_rates >= observed_best).mean()
print(f"\nNull distribution ({N_PERM} label-shuffles, {N_SPLITS_PERM} splits each):")
print(f"  mean={perm_max_rates.mean():.2f}  max={perm_max_rates.max():.2f}")
print(f"P(best of 50 random genes >= {observed_best}) = {p_value:.3f}")

perm 1/20 done, best rate this perm: 0.30
perm 2/20 done, best rate this perm: 0.20
perm 3/20 done, best rate this perm: 0.40
perm 4/20 done, best rate this perm: 0.30
perm 5/20 done, best rate this perm: 0.20
perm 6/20 done, best rate this perm: 0.30
perm 7/20 done, best rate this perm: 0.20
perm 8/20 done, best rate this perm: 0.30
perm 9/20 done, best rate this perm: 0.30
perm 10/20 done, best rate this perm: 0.30
perm 11/20 done, best rate this perm: 0.40
perm 12/20 done, best rate this perm: 0.40
perm 13/20 done, best rate this perm: 0.40
perm 14/20 done, best rate this perm: 0.50
perm 15/20 done, best rate this perm: 0.30
perm 16/20 done, best rate this perm: 0.30
perm 17/20 done, best rate this perm: 0.30
perm 18/20 done, best rate this perm: 0.30
perm 19/20 done, best rate this perm: 0.20
perm 20/20 done, best rate this perm: 0.20

Null distribution (20 label-shuffles, 10 splits each):
  mean=0.30  max=0.50
P(best of 50 random genes >= 0.7) = 0.000
